## RAG Implementation
### By: Emanda Bisrat 

#### Imports

In [ ]:
#I had to pip install the following packages: langchain, pypdf, -U langchain-community, sentence-transformers

In [39]:
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
from transformers import pipeline

#### Load Data

In [40]:
loader1 = PyPDFLoader("agentic_systems.pdf")
loader2 = PyPDFLoader("generative_vision.pdf")

docs1 = loader1.load()
docs2 = loader2.load()

documents = docs1 + docs2

#### Verify Data loaded correctly

In [41]:
print(len(documents))
print(documents[0].page_content[:500])

84
Agentic Systems: A Guide to Transforming
Industries with Vertical AI Agents
Fouad Bousetouane1,2
1The University of Chicago, USA
22ndsight.ai
bousetouane@uchicago.edu
Abstract
The evolution of agentic systems represents a significant milestone
in artificial intelligence and modern software systems, driven by the
demand for vertical intelligence tailored to diverse industries. These
systems enhance business outcomes through adaptability, learning,
and interaction with dynamic environments. At the


In [42]:
print(documents[1].page_content[:500])

Contents
1 Introduction 3
1.1 The Shortcomings of Traditional SaaS Platforms . . . . . . . . 3
1.2 The Transition to Context-Aware Systems . . . . . . . . . . . 4
2 The Rise of Vertical AI Agent Solutions 5
2.1 Operational Advantages of Vertical AI Agents . . . . . . . . . 5
2.1.1 1. Targeted Domain Expertise . . . . . . . . . . . . . . 5
2.1.2 2. Dynamic Adaptability in Real-Time Operations . . 6
2.1.3 3. End-to-End Workflow Automation . . . . . . . . . . 6
3 What Are LLM Agents? 7
3.1 Definiti


#### Chunk Documents

In [43]:

#Set overlap to 50 to preserve the meaning across the boundaries 
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

#### Verify Data Chunks

In [44]:
print(len(chunks))
print(chunks[0].page_content)

456
Agentic Systems: A Guide to Transforming
Industries with Vertical AI Agents
Fouad Bousetouane1,2
1The University of Chicago, USA
22ndsight.ai
bousetouane@uchicago.edu
Abstract
The evolution of agentic systems represents a significant milestone
in artificial intelligence and modern software systems, driven by the
demand for vertical intelligence tailored to diverse industries. These


#### Create Embeddings

In [45]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
#embedding_model = HuggingFaceEmbeddings(
#    model_name="BAAI/bge-base-en-v1.5"
#)

#### Store in Chroma

In [46]:
vectordb = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="./chroma_db_v12"
)

vectordb.persist()

In [47]:
print(vectordb._collection.count())

456


The vector database equals the amount of chunks 

#### Create Retriever

In [48]:
retriever = vectordb.as_retriever(search_kwargs={"k": 4})

#### Test Retrieval 

In [29]:
query = "What are vertical AI agents?"

docs = retriever.get_relevant_documents(query)

In [31]:
def clean_chunks(docs):
    cleaned = []
    for d in docs:
        text = d.page_content.lower()
        if (
            "arxiv" in text
            or "references" in text
            or "wasserstein" in text
            or "[1]" in text
            or "[2]" in text
        ):
            continue
        cleaned.append(d)
    return cleaned

docs = clean_chunks(docs)

In [32]:
for i, d in enumerate(docs):
    print(f"\n Chunk {i}")
    print(d.page_content[:500])


 Chunk 0
Vertical Ai agent solutions are rapidly gaining momentum, with major
players such as Google, AWS, OpenAI, and Microsoft spearheading efforts
to develop platforms that simplify and scale the creation of vertical AI so-
lutions. While these advancements signal a transformative shift, we are still
in the early stages of this journey, with operational patterns only begin-

 Chunk 1
These specialized capabilities make vertical AI agents indispensable in fields
where accuracy, reliability, and regulatory adherence are critical.
2.1.2 2. Dynamic Adaptability in Real-Time Operations
Unlike traditional systems, vertical AI agents excel in dynamic environ-
ments, continuously adapting to changing conditions and operational de-
mands. They achieve this through:

 Chunk 2
in the early stages of this journey, with operational patterns only begin-
ning to take shape. These emerging platforms aim to provide standardized
frameworks for fine-tuning, deployment, and integration, enabling a mor

#### Build the LLM using FLAN-T5

In [69]:
'''generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    device=-1,  
    max_new_tokens=128,
    truncation=True
)''' 


Device set to use cpu


In [49]:
generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    device=-1,
    max_new_tokens=128,
    repetition_penalty=1.5,
    temperature=0.7,
    top_p=0.9,
    no_repeat_ngram_size=3
)

Device set to use cpu


In [36]:
generator("Explain what AI is in one sentence.")

[{'generated_text': 'artificial intelligence'}]

In [37]:
generator("Explain what artificial intelligence is in one complete sentence.")

[{'generated_text': 'artificial intelligence (AI) is a type of artificial intelligence.'}]

In [38]:
generator("What is artificial intelligence? Give a clear explanation.")

[{'generated_text': 'Artificial Intelligence (AI) is the development of artificial intelligence. Artificial intelligence is a type of technology that can be applied to many different areas of the human body. The term AI refers to the use of computer algorithms to learn new things. The answer: AI.'}]

#### Build RAG Propmt and Function

In [50]:
def build_prompt(query, context):
    return f"""
You are an expert AI researcher.

Your task is to answer the question using ONLY the context below.

Do NOT copy the context directly.
Instead, extract the relevant information and summarize it.

Write a clear and complete answer in 2-4 sentences.

If the answer cannot be found, say: "not found in the provided text."

Context:
{context}

Question:
{query}

Answer:
"""

In [51]:
def rag_pipeline(query):
    docs = retriever.get_relevant_documents(query)
    context = "\n\n".join([d.page_content for d in docs[:3]])
    prompt = build_prompt(query, context)
    output = generator(prompt)[0]['generated_text']

    return {
        "question": query,
        "contexts": [d.page_content for d in docs],
        "answer": output
    }

#### Test Function

In [93]:
result = rag_pipeline("What are the core architectural layers of a vertical AI agent system?")
print(result["answer"])

not found in the given text.


In [100]:
rag_pipeline("What are vertical AI agents?")

{'question': 'What are vertical AI agents?',
 'contexts': ['Vertical Ai agent solutions are rapidly gaining momentum, with major\nplayers such as Google, AWS, OpenAI, and Microsoft spearheading efforts\nto develop platforms that simplify and scale the creation of vertical AI so-\nlutions. While these advancements signal a transformative shift, we are still',
  'These specialized capabilities make vertical AI agents indispensable in fields\nwhere accuracy, reliability, and regulatory adherence are critical.\n2.1.2 2. Dynamic Adaptability in Real-Time Operations\nUnlike traditional systems, vertical AI agents excel in dynamic environ-',
  'arXiv:2204.14198, 2022.\n[2] Martin Arjovsky, Soumith Chintala, and L´ eon Bottou. Wasserstein gan.\narXiv preprint arXiv:1701.07875 , 2017.\n[3] Fouad Bousetouane. Agentic systems: A guide to transforming indus-\ntries with vertical ai agents, 2025.',
  '2 The Rise of Vertical AI Agent Solutions\nAs industries face increasingly complex and domain-spec

#### RAGAS Evaluation 

In [52]:
queries = [
    "What are the core architectural layers of a vertical AI agent system as described by Dr. Bousetouane?",
    "Explain how agentic systems are designed to improve decision-making in enterprise workflows.",
    "What are the main advantages of using diffusion models over GANs for image generation?",
    "Describe a real-world use case of vision-language models mentioned in the paper by Dr. Bousetouane.",
    "How can vision-based generative models be integrated into agentic systems for task automation?"
]

In [53]:
results = [rag_pipeline(q) for q in queries]

In [54]:
for r in results:
    print("\nQUESTION:", r["question"])
    print("ANSWER:", r["answer"])


QUESTION: What are the core architectural layers of a vertical AI agent system as described by Dr. Bousetouane?
ANSWER: LLM agents, leveraging large In response to the need for consistency and scalability, this work at- tempts to define a level of standardization for Vertical AI agent design patterns by identifying core building blocks and proposing a Cogni- tive Skills Module, which incorporates domain-specific, purpose- built inference capabilities.

QUESTION: Explain how agentic systems are designed to improve decision-making in enterprise workflows.
ANSWER: not found in the given text.

QUESTION: What are the main advantages of using diffusion models over GANs for image generation?
ANSWER: Iterative nature also enables diffusion mod-els to produce more diverse samples, making them particularly effective in applications requiring fine detail and variety.

QUESTION: Describe a real-world use case of vision-language models mentioned in the paper by Dr. Bousetouane.
ANSWER: Bootstrapp

#### Reference Answers

In [55]:
reference_answers = [
    "Vertical AI agent systems are composed of several architectural layers including a perception layer for data ingestion, a reasoning layer for decision-making, a memory layer for storing historical context, and an action layer responsible for executing tasks and interacting with external systems.",
    
    "Agentic systems improve decision-making in enterprise workflows by combining contextual awareness, real-time data processing, and autonomous reasoning. They enable dynamic adaptation, allowing organizations to automate complex workflows and make informed decisions with minimal human intervention.",
    
    "Diffusion models offer advantages over GANs such as more stable training, reduced risk of mode collapse, and the ability to generate diverse and high-quality outputs through iterative denoising processes.",
    
    "A real-world use case of vision-language models includes applications such as image captioning and medical imaging analysis, where models interpret visual data and generate meaningful textual descriptions to assist human decision-making.",
    
    "Vision-based generative models can be integrated into agentic systems by enabling perception capabilities, allowing agents to interpret visual environments and automate tasks such as monitoring, simulation, and decision-making in complex real-world scenarios."
]

#### RAGAS

In [56]:
from datasets import Dataset

data = {
    "question": [r["question"] for r in results],
    "answer": [r["answer"] for r in results],
    "contexts": [r["contexts"] for r in results],
    "ground_truth": reference_answers
}

dataset = Dataset.from_dict(data)

In [57]:
from ragas.embeddings import LangchainEmbeddingsWrapper

ragas_embeddings = LangchainEmbeddingsWrapper(embedding_model)

/var/folders/f4/z7nrrfjs70jfzklg1786wdp80000gn/T/ipykernel_98567/1715043724.py:3: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(embedding_model)


In [25]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall

/var/folders/f4/z7nrrfjs70jfzklg1786wdp80000gn/T/ipykernel_80428/2431867005.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
/var/folders/f4/z7nrrfjs70jfzklg1786wdp80000gn/T/ipykernel_80428/2431867005.py:2: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
/var/folders/f4/z7nrrfjs70jfzklg1786wdp80000gn/T/ipykernel_80428/2431867005.py:2: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'raga

In [58]:
scores = evaluate(
    dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall
    ],
    embeddings=ragas_embeddings   # 👈 ADD THIS
)
print(scores)

Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


{'faithfulness': 0.7167, 'answer_relevancy': 0.3143, 'context_precision': 0.4667, 'context_recall': 0.6000}


In [72]:
scores_df["question"] = [r["question"] for r in results]
scores_df["question"]

0    What are the core architectural layers of a ve...
1    Explain how agentic systems are designed to im...
2    What are the main advantages of using diffusio...
3    Describe a real-world use case of vision-langu...
4    How can vision-based generative models be inte...
Name: question, dtype: object

In [71]:
scores_df = scores.to_pandas()

summary = pd.DataFrame({
    "Query": [f"Q{i+1}" for i in range(len(results))],
    "Faithfulness": scores_df["faithfulness"].round(4),
    "Answer Relevancy": scores_df["answer_relevancy"].round(4),
    "Context Precision": scores_df["context_precision"].round(4),
    "Context Recall": scores_df["context_recall"].round(4)
})

display(summary)

# Averages row
averages = pd.DataFrame([{
    "Query": "Average",
    "Faithfulness": scores_df["faithfulness"].mean().round(4),
    "Answer Relevancy": scores_df["answer_relevancy"].mean().round(4),
    "Context Precision": scores_df["context_precision"].mean().round(4),
    "Context Recall": scores_df["context_recall"].mean().round(4)
}])

display(averages)

,Query,Faithfulness,Answer Relevancy,Context Precision,Context Recall
0,Q1,0.8333,0.0000,0.0000,0.0
1,Q2,0.0000,0.0000,0.3333,1.0
2,Q3,1.0000,0.6241,1.0000,1.0
3,Q4,1.0000,0.4593,0.0000,0.0
4,Q5,0.7500,0.4881,1.0000,1.0


,Query,Faithfulness,Answer Relevancy,Context Precision,Context Recall
0,Average,0.7167,0.3143,0.4667,0.6


### Evaluation

For query 1 (core architectural layers of a vertical AI agent system), the retriver did not do well at all. Both the context and precision and recall scored 0. There were no relevant chunks retrieved. The faithfulness was pretty high. 

For query 2 (agentic systems and enterprise decision-making), the context recall was very good but the faithfulness fell to 0. This is probably a hallucinated answer because the model didn't use the chunks. The relevancy is also 0. 

For query 3 (diffusion models vs GANs), the contect precision and recall are both really good (1.0), and the faithfulness is also very good (1.0). The relevancy was 0.62 which is not the best but still performs better than other queries. 

For query 4 (real-world use case of vision-language models), the fathfulness was very good but the context recall and precision were both 0. The answer relevancy (0.46) was moderate but not great. 

For query 5 (vision0-based generative models integrated into agentic systems), the context precision and recall are both 1.0. The faithfulness was also moderately good (0.75) and the answer relevancy was moderate (0.49). 

The summary of averages was:
1. Faithfulness 0.7167
2. Answer Relevancy 0.3143
3. Context Precision 0.4667
4. Context Recall 0.6000

Overall, the retrieval was pretty inconsistent across the five queries. Query 3 and 5 had perfect retrieval but query 4 and 1 had 0. They retrieved nothing useful. Answer relevancy was teh weakest which could be due to the small Flan-T5-base model. Faithfullness was the strongest metric so the model did stay within context for the most part and didn't resort to hallucination much. I think I should've definitely switched to a larger model like Mistral-7B to improve my metrics, particularly answer relevancy and faithfulness. 